# ExhibitFlow Dashboard — Colab
Chạy dashboard (video + map 2D + alerts) và mở link công khai qua Cloudflare Tunnel.

**Trước khi chạy:** bật GPU (Runtime → Change runtime type). Link chỉ sống khi cell cuối còn chạy.

In [ ]:
# 1. Cấu hình
REPO_URL = 'https://github.com/khangkaka066/ExhibitFlow.git'
BRANCH   = 'Hung'
ROOT     = '/content/ExhibitFlow'
WEIGHTS  = ''      # thư mục chứa 4 file trọng số (best_ckpt.pth, osnet_x0_25.pth, ltc_motion_mot17.pth, dma_gbm_mot17.gbm); có thể nhập lại trong Settings
DEMO     = False   # True = dữ liệu giả để xem thử giao diện (không cần GPU / trọng số)
PORT     = 8000
HF_TOKEN = ''      # KHÔNG ghi token vào notebook rồi push. Để trống: cell cuối sẽ hỏi khi chạy (không hiện ký tự)

In [ ]:
# 2. Clone repo + map 2D
import subprocess, os
def sh(c, **k): print('$', c); subprocess.run(c, shell=True, check=True, **k)
if not os.path.exists(ROOT):
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {ROOT}')
else:
    sh(f'cd {ROOT} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD')
FP = os.path.join(os.path.dirname(ROOT), 'multicam-2d-floorplan-tracker')
if not os.path.exists(FP):
    sh(f'git clone --depth 1 https://github.com/khangkaka066/multicam-2d-floorplan-tracker.git {FP}')
os.environ['FLOORPLAN_DIR'] = FP
PROJ = os.path.join(ROOT, 'exhibitflow_pipeline')

In [ ]:
# 3. Cài thư viện (~3-5 phút lần đầu)
sh(f'cd {PROJ} && python scripts/install_deps.py')
sh('pip install -q fastapi "uvicorn[standard]" python-multipart psutil')

In [ ]:
# 4. Chạy dashboard + tunnel  ->  bấm vào LINK CÔNG KHAI in ra bên dưới (gửi cho mọi người)
import os, subprocess, sys
subprocess.run(['pkill', '-f', 'dashboard.server'])   # dọn dashboard / tunnel còn sót từ lần chạy trước
subprocess.run(['pkill', '-f', 'cloudflared'])
env = dict(os.environ)
if WEIGHTS: env['EF_WEIGHTS_DIR'] = WEIGHTS
import getpass
token = HF_TOKEN or os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN (Enter để bỏ qua, tải sẽ chậm hơn): ').strip()
if token:
    env['HF_TOKEN'] = token          # huggingface_hub tự đọc biến này khi tải GGUF / Grounding-DINO
args = [sys.executable, '-u', '-m', 'dashboard.launch', '--port', str(PORT), '--data-dir', os.path.join(PROJ, 'dashboard_data')] + (['--demo'] if DEMO else [])
subprocess.run(args, cwd=PROJ, env=env)